# ระดับ 1 — Zero-shot inference ของ CFB (FFDBackbone) บน DeepFakeFace

**Paper:** Guo et al., *Revisiting Face Forgery Detection: From Facial Representation to Forgery Detection*, IEEE TPAMI 2026
**Code:** https://github.com/zhenglab/FFDBackbone

**เป้าหมาย:** ใช้ checkpoint ที่ผู้เขียนเทรนบน FF++ c23 (ไม่ fine-tune เพิ่ม) ทดสอบกับภาพปลอมจาก generator
ที่โมเดลไม่เคยเห็น (unseen domain) ในชุด **DeepFakeFace (DFF)** แล้ววัด AUC / ACC / EER แยกตาม generator

| subset ใน DFF | generator | ชนิด |
|---|---|---|
| `wiki` | IMDB-WIKI (ภาพจริง) | real |
| `text2img` | Stable Diffusion v1.5 | diffusion (สร้างทั้งภาพ) |
| `inpainting` | Stable Diffusion Inpainting | diffusion (แก้เฉพาะบริเวณหน้า) |
| `insight` | InsightFace | face swap |

**ลำดับใน notebook**
1. ตั้งค่า → ติดตั้ง → clone repo + patch ให้รันกับ library รุ่นใหม่
2. โหลด checkpoint + ตรวจว่า weight เข้าครบ
3. Sanity check: (ก) ภาพตัวอย่างใน repo (ข) คำนวณ AUC จากไฟล์ผลของผู้เขียนเทียบกับ paper
4. โหลด DFF → crop หน้าด้วย RetinaFace (pipeline เดียวกับ repo) → inference
5. Metric แยกตาม generator + bootstrap 95% CI, ROC, histogram
6. วิเคราะห์ threshold (τ=0.5 vs τ_ot vs τ_ot^UTOM ตาม Eq. 10–11 ของ paper)

**ตั้งค่า Kaggle ก่อนรัน:** Accelerator = GPU (T4 ×2 หรือ P100) · Internet = On
ใช้เวลารวมประมาณ 20–40 นาทีที่ `N_PER_CLASS = 3000`

## 0. ตั้งค่า

In [ ]:
import os, sys, re, time, json, glob, random, shutil, zipfile, hashlib, subprocess, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
T_START = time.time()

ON_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("FFD_WORK", "./work")).resolve()
TMP_DIR = Path("/tmp/ffd") if ON_KAGGLE else WORK_DIR / "tmp"   # ไฟล์ใหญ่ (zip, weights, repo) ไม่ให้ไปปนใน output
REPO_DIR = TMP_DIR / "FFDBackbone"
ZIP_DIR = TMP_DIR / "dff_zips"
RAW_DIR = TMP_DIR / "dff_raw"
WEIGHT_DIR = TMP_DIR / "weights"
CROP_DIR = WORK_DIR / "dff_crops"          # หน้า 224×224 ที่ crop แล้ว (เก็บไว้ใช้ซ้ำ / อัปเป็น Kaggle Dataset ได้)
OUT_DIR = WORK_DIR / "results_level1"
for d in [TMP_DIR, ZIP_DIR, RAW_DIR, WEIGHT_DIR, CROP_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/zhenglab/FFDBackbone.git"
GDRIVE_FOLDER = "https://drive.google.com/drive/folders/1jFIpb4TftJiL82h69c3Roqy8uIzGQP4G"

# --- checkpoint ---
# ถ้าอัป checkpoint เป็น Kaggle Dataset ไว้แล้ว ให้ใส่ path ตรงนี้ เช่น "/kaggle/input/ffd-ours/Final.tar"
# ถ้าเป็น None จะค้นใน /kaggle/input ก่อน แล้วค่อยดาวน์โหลดจาก Google Drive
CKPT_PATH = None
CKPT_NAME_HINTS = ["ours", "final", "ddbf", "dbbf"]   # ใช้เลือกไฟล์จาก Google Drive (ไม่สนตัวพิมพ์เล็ก/ใหญ่)
CKPT_EXCLUDE = ["beit-1k-face-55w"]                    # ตัวนี้คือ pre-trained backbone (ยังไม่ fine-tune) ไม่ใช่ตัว detector

# --- data ---
HF_REPO = "OpenRL/DeepFakeFace"
SUBSETS = {            # ชื่อ subset : (ไฟล์ zip, label 0=real 1=fake, ชื่อที่ใช้ในรายงาน)
    "wiki":       ("wiki.zip",       0, "Real (IMDB-WIKI)"),
    "text2img":   ("text2img.zip",   1, "SD v1.5 (text2img)"),
    "inpainting": ("inpainting.zip", 1, "SD Inpainting"),
    "insight":    ("insight.zip",    1, "InsightFace (swap)"),
}
N_PER_CLASS = 3000     # จำนวนภาพต่อ subset (None = ใช้ทั้งหมด 30,000 ต่อ subset)
DELETE_ZIPS_AFTER = True

# --- preprocessing / inference (ค่าตาม repo) ---
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
DET_THRES = 0.5        # threshold ของ RetinaFace ใน get_valid_faces()
BATCH_SIZE = 64
NUM_WORKERS = 2
USE_AMP = True
USE_FUSION = "auto"    # "auto" = ถ้า checkpoint มี branch ที่ 2 ให้คำนวณผลแบบ fuse ด้วย (ใช้เปรียบเทียบ), False = main branch อย่างเดียว

# --- analysis ---
UTOM_LAMBDA = 0.02     # λ ใน p' = tanh(λp / 2(u+ε))
UTOM_EPS = 1e-7
N_BOOT = 1000          # จำนวน bootstrap สำหรับ 95% CI ของ AUC
SEED = 42

random.seed(SEED)
print("ON_KAGGLE:", ON_KAGGLE, "| WORK_DIR:", WORK_DIR, "| TMP_DIR:", TMP_DIR)

## 1. ติดตั้ง library
ไม่ใช้ `requirements.txt` ของ repo ตรง ๆ เพราะ pin เวอร์ชันเก่า (numpy 1.23, torch 1.13, timm 0.9.2)
ซึ่งจะทำให้ environment ของ Kaggle พัง — ใช้ของที่ Kaggle มีอยู่แล้ว แล้ว patch โค้ด repo แทน (cell ถัดไป)

In [ ]:
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=False)

pip_install("gdown")
try:
    import timm  # noqa
except ImportError:
    pip_install("timm")
try:
    import huggingface_hub  # noqa
except ImportError:
    pip_install("huggingface_hub")

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
import timm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENV = {
    "python": sys.version.split()[0], "torch": torch.__version__, "timm": timm.__version__,
    "numpy": np.__version__, "opencv": cv2.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
}
print(ENV)

## 2. Clone repo + patch
Patch ที่ทำ (ไม่แตะ logic ของโมเดล):
- `np.float` / `np.int` / `np.bool` → `np.float64` / `np.int64` / `np.bool_` (ถูกลบตั้งแต่ numpy 1.24)
- `np.reshape(..., newshape=...)` → positional (เลิกใช้ใน numpy 2.x)
- เพิ่ม `import torch.utils.model_zoo` ใน RetinaFace loader (torch รุ่นใหม่ไม่ import ให้อัตโนมัติ) และโหลด weight ของ RetinaFace ลง CPU ก่อน

In [ ]:
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip()
print("repo commit:", REPO_COMMIT)

def patch_repo(repo: Path):
    n_files = 0
    for py in list((repo / "face_process").rglob("*.py")) + list((repo / "models" / "lib" / "BEiT_v2").rglob("*.py")):
        src = py.read_text(encoding="utf-8", errors="ignore")
        new = re.sub(r"np\.float\b", "np.float64", src)
        new = re.sub(r"np\.int\b", "np.int64", new)
        new = re.sub(r"np\.bool\b", "np.bool_", new)
        new = new.replace("np.reshape(b, newshape=(1, 2))", "np.reshape(b, (1, 2))")
        if py.name == "alignment.py":
            if "import torch.utils.model_zoo" not in new:
                new = "import torch.utils.model_zoo\n" + new
            new = new.replace("model_zoo.load_url(url)", "model_zoo.load_url(url, map_location='cpu')")
        if new != src:
            py.write_text(new, encoding="utf-8")
            n_files += 1
    return n_files

print("patched files:", patch_repo(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import importlib.util

def load_module_from_file(name, path):
    """โหลดไฟล์ .py ตรง ๆ — ถ้า import ผ่าน package `models` จะไปเรียก models/__init__.py ซึ่ง import
    backbone ทุกตัว (ต้องใช้ mmpretrain, yacs ฯลฯ ที่ไม่จำเป็นสำหรับงานนี้)"""
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

modeling_finetune = load_module_from_file(                  # BEiT v2 ViT-B/16 ของ repo
    "beitv2_modeling_finetune", REPO_DIR / "models" / "lib" / "BEiT_v2" / "modeling_finetune.py")
from face_process.lib.ct.detection import FaceDetector     # RetinaFace (mobilenet0.25) แบบ FTCN
from face_process import face_utils                         # crop_aligned (ArcFace 5-point alignment)

## 3. Checkpoint
ลำดับการหา checkpoint: `CKPT_PATH` → ไฟล์ใน `/kaggle/input` → ดาวน์โหลดจาก Google Drive ด้วย `gdown`

ถ้า gdown ล้มเหลว (Google Drive จำกัดโควตาดาวน์โหลดบ่อย) ให้ดาวน์โหลดไฟล์ checkpoint ของ **Ours** เองจาก
Google Drive → Kaggle: *Add Input → Upload → New Dataset* → ใส่ path ใน `CKPT_PATH` แล้วรันใหม่

In [ ]:
CKPT_EXTS = (".tar", ".pth", ".pt", ".ckpt", ".bin")

def _is_candidate(name: str) -> bool:
    low = name.lower()
    return low.endswith(CKPT_EXTS) and not any(x in low for x in CKPT_EXCLUDE)

def _rank(name: str) -> int:
    low = name.lower()
    return sum(h in low for h in CKPT_NAME_HINTS)

def find_or_download_ckpt():
    if CKPT_PATH and Path(CKPT_PATH).exists():
        return Path(CKPT_PATH)
    # 1) Kaggle input / ไฟล์ที่เคยดาวน์โหลดไว้
    local = [p for root in ["/kaggle/input", str(WEIGHT_DIR)] if Path(root).exists()
             for p in Path(root).rglob("*") if p.is_file() and _is_candidate(p.name)]
    if local:
        local.sort(key=lambda p: (-_rank(str(p)), str(p)))
        print("พบ checkpoint ในเครื่อง:", *local, sep="\n  ")
        return local[0]
    # 2) Google Drive
    import gdown
    print("กำลังอ่านรายชื่อไฟล์ใน Google Drive folder ...")
    import inspect
    kw = dict(url=GDRIVE_FOLDER, skip_download=True, quiet=True)
    if "remaining_ok" in inspect.signature(gdown.download_folder).parameters:   # gdown รุ่นเก่า (<5) จำกัด 50 ไฟล์
        kw["remaining_ok"] = True
    files = gdown.download_folder(**kw)
    entries = []
    for f in files or []:
        fid = getattr(f, "id", None) or (f[0] if isinstance(f, (tuple, list)) else None)
        fpath = getattr(f, "path", None) or (f[1] if isinstance(f, (tuple, list)) else "")
        entries.append((fid, str(fpath)))
    print("ไฟล์ใน Drive:", *[p for _, p in entries], sep="\n  ")
    cands = [(fid, p) for fid, p in entries if _is_candidate(p)]
    if not cands:
        raise RuntimeError("ไม่พบไฟล์ checkpoint ใน Drive — ดาวน์โหลดเองแล้วตั้ง CKPT_PATH")
    cands.sort(key=lambda x: (-_rank(x[1]), x[1]))
    fid, rel = cands[0]
    out = WEIGHT_DIR / rel
    out.parent.mkdir(parents=True, exist_ok=True)
    print(f"ดาวน์โหลด: {rel}")
    gdown.download(id=fid, output=str(out), quiet=False)
    if not out.exists() or out.stat().st_size < 10_000_000:
        raise RuntimeError("ดาวน์โหลดไม่สำเร็จ (อาจติดโควตา Google Drive) — ดาวน์โหลดเองแล้วตั้ง CKPT_PATH")
    return out

def sha256_of(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

ckpt_file = find_or_download_ckpt()
CKPT_SHA = sha256_of(ckpt_file)
print("checkpoint:", ckpt_file, f"({ckpt_file.stat().st_size/1e6:.0f} MB)  sha256:", CKPT_SHA[:16])

### 3.1 สร้างโมเดลและโหลด weight
โครงสร้างเหมือน `DDBF_BEiT_v2` ใน `models/custom.py` (ชื่อ layer ตรงกันเพื่อให้ state dict เข้าได้ตรง ๆ):
- **main branch** = `backbone_model` → LayerNorm → FC — ใช้ตอน inference ตาม paper และ `eval.py`
- **auxiliary branch** = `backbone_model2` (ใช้เฉพาะตอนเทรน) — ถ้า checkpoint มี จะคำนวณผลแบบ uncertainty fusion
  `w = softmax(-10·[u₁, u₂])` เพิ่มเพื่อเปรียบเทียบ
- uncertainty `u = K/S`, `S = Σ(exp(clamp(logit, -10, 10)) + 1)`, K = 2 (Eq. 3–4)

⚠️ `inference.py` ของ repo เก็บเฉพาะ key ที่ขึ้นต้นด้วย `module.` และโหลดแบบ `strict=False` — ถ้า checkpoint ไม่มี prefix นี้
โมเดลจะได้ weight สุ่มโดยไม่มี error ใด ๆ  ใน notebook นี้จึงตรวจว่า **ทุก key ของ main branch ต้องโหลดเข้าได้ครบ**

In [ ]:
BEIT_CONFIG = dict(drop_path_rate=0.1, use_mean_pooling=False, init_values=0.1, qkv_bias=True,
                   use_abs_pos_emb=False, use_rel_pos_bias=True, use_shared_rel_pos_bias=False)

def make_beit():
    m = modeling_finetune.beit_base_patch16_224(**BEIT_CONFIG)
    m.head = nn.Identity()
    return m

def evidential_uncertainty(logits, num_classes=2):
    alpha = torch.exp(torch.clamp(logits, -10, 10)) + 1
    return num_classes / alpha.sum(dim=1, keepdim=True)

class CFBDetector(nn.Module):
    def __init__(self, dual: bool):
        super().__init__()
        self.dual = dual
        self.backbone_model = make_beit()
        if dual:
            self.backbone_model2 = make_beit()
        self.norm_for_cls = nn.LayerNorm(768)
        self.header = nn.Sequential(nn.Linear(768, 2))

    def classify(self, feat):
        return self.header(self.norm_for_cls(feat))

    def forward(self, x):                        # x: [B, 3, 224, 224]
        f1 = self.backbone_model(x)
        out = {"main": self.classify(f1)}
        if self.dual:
            f2 = self.backbone_model2(x)
            l2 = self.classify(f2)
            u = torch.cat([evidential_uncertainty(out["main"].float()), evidential_uncertainty(l2.float())], dim=-1)
            w = torch.softmax(-10 * u, dim=-1).to(f1.dtype)
            out["branch2"] = l2
            out["fused"] = self.classify(w[:, :1] * f1 + w[:, 1:] * f2)
        return out

def load_state_dict_any(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    meta = {k: v for k, v in ck.items() if not isinstance(v, (dict, torch.Tensor))} if isinstance(ck, dict) else {}
    sd = ck
    for key in ("state_dict", "model", "model_state_dict", "net"):
        if isinstance(sd, dict) and isinstance(sd.get(key), dict):
            sd = sd[key]
            break
    sd = {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items() if torch.is_tensor(v)}
    return sd, meta

state_dict, ckpt_meta = load_state_dict_any(ckpt_file)
HAS_BRANCH2 = any(k.startswith("backbone_model2.") for k in state_dict)
DUAL = HAS_BRANCH2 and USE_FUSION in ("auto", True)
print("checkpoint meta:", {k: ckpt_meta[k] for k in list(ckpt_meta)[:8]})
print("มี auxiliary branch:", HAS_BRANCH2, "| คำนวณ fused output:", DUAL)

model = CFBDetector(dual=DUAL)
expected = set(model.state_dict().keys())
loadable = {k: v for k, v in state_dict.items() if k in expected}
missing = sorted(expected - set(loadable))
unexpected = sorted(set(state_dict) - expected)
shape_bad = [k for k, v in loadable.items() if v.shape != model.state_dict()[k].shape]
print(f"keys: expected={len(expected)} loaded={len(loadable)} missing={len(missing)} unexpected={len(unexpected)}")
if missing or shape_bad:
    raise RuntimeError(f"checkpoint ไม่ตรงกับโมเดล (อาจเลือกไฟล์ผิด เช่น pre-trained backbone)\n"
                       f"missing[:10]={missing[:10]}\nshape mismatch={shape_bad[:10]}")
if unexpected:
    print("unexpected keys (ไม่ใช้):", unexpected[:10], "..." if len(unexpected) > 10 else "")
model.load_state_dict(loadable, strict=True)
model = model.to(DEVICE).eval()
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"โหลดครบ ✓  parameters = {n_params:.1f}M  device = {DEVICE}")
del state_dict, loadable

## 4. Face preprocessing (เหมือน `face_process/real_face_process_Frame.py`)
1. RetinaFace (mobilenet0.25) หาใบหน้า เก็บเฉพาะ score ≥ 0.5 แล้วเลือกหน้าแรก
2. Align ด้วย 5 landmark (similarity transform ไปยัง template ArcFace) ได้ภาพ 224×224
3. Crop ตาม bbox ที่ถูกหมุนแล้ว + margin w/8, h/8 ต่อด้าน (`crop_face_sbi`, phase='test') → resize 224×224
4. Normalize ด้วย ImageNet mean/std

⚠️ `inference.py` ของ repo ส่งภาพ **BGR** (จาก `cv2.imread`) เข้าโมเดลตรง ๆ แต่ตอนเทรน/ประเมิน (`Image_dataset*.py`)
อ่านภาพด้วย PIL ซึ่งเป็น **RGB** → notebook นี้แปลงเป็น RGB ให้ตรงกับตอนเทรน (ดูผลเทียบใน sanity check)

In [ ]:
detector = FaceDetector(0 if DEVICE.type == "cuda" else -1)

def valid_faces(faces, max_count=10, thres=DET_THRES):
    """เหมือน get_valid_faces() ของ repo สำหรับภาพเดียว"""
    out = []
    for face in faces[:max_count]:
        box, lm, score = face
        if score < thres:
            continue
        out.append((box.astype(np.float64), lm.astype(np.float64), float(score)))
    return out

def crop_face_test(img, bbox):
    """crop_face_sbi(..., margin=False, crop_by_bbox=True, phase='test') ของ repo"""
    H, W = img.shape[:2]
    (x0, y0), (x1, y1) = bbox
    w, h = x1 - x0, y1 - y0
    w0 = w1 = w / 4 * 0.5
    h0 = h1 = h / 4 * 0.5
    y0n, y1n = max(0, int(y0 - h0)), min(H, int(y1 + h1) + 1)
    x0n, x1n = max(0, int(x0 - w0)), min(W, int(x1 + w1) + 1)
    return img[y0n:y1n, x0n:x1n]

def crop_face(path):
    """คืน (crop RGB 224×224 หรือ None, ข้อมูลประกอบ)"""
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)        # BGR
    if img is None or min(img.shape[:2]) < 16:
        return None, {"status": "read_error", "n_faces": 0, "det_score": np.nan}
    try:
        faces = valid_faces(detector.detect(img))
    except Exception as e:  # ภาพแปลก ๆ บางภาพทำให้ detector error
        return None, {"status": f"det_error:{type(e).__name__}", "n_faces": 0, "det_score": np.nan}
    if not faces:
        return None, {"status": "no_face", "n_faces": 0, "det_score": np.nan}
    bbox, lm5, score = faces[0]
    aligned, _, bb = face_utils.crop_aligned(img, lm5, landmarks_68=None, bboxes=bbox, aligned_image_size=IMG_SIZE)
    crop = crop_face_test(aligned, np.array([[bb[0], bb[1]], [bb[2], bb[3]]]))
    if crop.size == 0:
        return None, {"status": "empty_crop", "n_faces": len(faces), "det_score": score}
    crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    is_gray = bool(np.allclose(crop[..., 0], crop[..., 1], atol=2) and np.allclose(crop[..., 1], crop[..., 2], atol=2))
    return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB), {"status": "ok", "n_faces": len(faces),
                                                   "det_score": score, "is_gray": is_gray}

def to_tensor(rgb):
    x = (rgb.astype(np.float32) / 255.0 - np.array(MEAN, np.float32)) / np.array(STD, np.float32)
    return torch.from_numpy(x.transpose(2, 0, 1).copy())

@torch.no_grad()
def predict_batch(x):
    x = x.to(DEVICE, non_blocking=True)
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP and DEVICE.type == "cuda"):
        out = model(x)
    res = {}
    for name, logits in out.items():
        logits = logits.float()
        res[f"p_{name}"] = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        res[f"u_{name}"] = evidential_uncertainty(logits)[:, 0].cpu().numpy()
        res[f"logit_{name}"] = (logits[:, 1] - logits[:, 0]).cpu().numpy()
    return res

## 5. Sanity check
### 5.1 ภาพตัวอย่างใน repo (`test_img/`)
เทียบกับ `inference_results.csv` ที่ผู้เขียนแนบมา — ถ้า weight โหลดถูก ภาพ Fake ควรได้ score สูง ภาพ Real ควรได้ต่ำ
(คอลัมน์ `p_BGR` จำลองวิธีของ `inference.py` ต้นฉบับ, `p_RGB` คือวิธีที่ใช้ใน notebook นี้)

In [ ]:
ref = pd.read_csv(REPO_DIR / "inference_results.csv")
ref["file"] = ref["img_name"].map(lambda s: Path(s).name)
rows = []
for p in sorted((REPO_DIR / "test_img").glob("*.png")):
    if "aligned" in p.name:
        continue
    rgb, info = crop_face(p)
    if rgb is None:
        rows.append({"file": p.name, "status": info["status"]})
        continue
    x = torch.stack([to_tensor(rgb), to_tensor(rgb[..., ::-1].copy())])
    pr = predict_batch(x)["p_main"]
    rows.append({"file": p.name, "status": info["status"], "p_RGB": pr[0], "p_BGR": pr[1]})
sanity = pd.DataFrame(rows).merge(ref[["file", "y_pred"]].rename(columns={"y_pred": "p_repo_csv"}), on="file", how="left")
sanity["label"] = np.where(sanity["file"].str.startswith("Fake"), "fake", "real")
print(sanity.round(4).to_string(index=False))
sanity.to_csv(OUT_DIR / "sanity_test_img.csv", index=False)

### 5.2 คำนวณ metric จากไฟล์ผลรายวิดีโอของผู้เขียน
repo แนบ `checkpoints/Ours/Ours/test/*.csv` (score รายวิดีโอ = ค่าเฉลี่ย 8 frame) — คำนวณ video-level AUC ใหม่เพื่อยืนยันตัวเลขใน paper
(CDF 90.46, DFDC 84.90, FFIW 90.97, FF++ 99.36) ก่อนจะทำระดับ 2 ด้วยตัวเอง

In [ ]:
def eer_score(y, s):
    fpr, tpr, _ = roc_curve(y, s)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[i] + fnr[i]) / 2)

def basic_metrics(y, s, thr=0.5):
    y, s = np.asarray(y).astype(int), np.asarray(s, dtype=float)
    pred = (s >= thr).astype(int)
    tnr = float(((pred == 0) & (y == 0)).sum() / max((y == 0).sum(), 1))
    tpr = float(((pred == 1) & (y == 1)).sum() / max((y == 1).sum(), 1))
    return {
        "n_real": int((y == 0).sum()), "n_fake": int((y == 1).sum()),
        "AUC": roc_auc_score(y, s), "AP": average_precision_score(y, s),
        "EER": eer_score(y, s), "ACC@0.5": float((pred == y).mean()),
        "BalACC@0.5": (tnr + tpr) / 2, "RealAcc(TNR)": tnr, "FakeAcc(TPR)": tpr,
    }

paper_auc = {"FaceForensics_c23": 99.36, "Celeb-DF": 90.46, "DFDC": 84.90, "FFIW": 90.97}
rows = []
for name, auc_ref in paper_auc.items():
    f = REPO_DIR / "checkpoints/Ours/Ours/test" / f"{name}.csv"
    if not f.exists():
        continue
    d = pd.read_csv(f)
    m = basic_metrics(d["label"], d["predict"])
    rows.append({"dataset": name, "n_videos": len(d), "AUC (คำนวณใหม่)": 100 * m["AUC"], "AUC (paper)": auc_ref,
                 "ACC@0.5": 100 * m["ACC@0.5"], "EER": 100 * m["EER"]})
author_check = pd.DataFrame(rows)
print(author_check.round(2).to_string(index=False))
author_check.to_csv(OUT_DIR / "author_csv_check.csv", index=False)

## 6. DeepFakeFace: ดาวน์โหลด → สุ่มตัวอย่าง → crop หน้า
- แต่ละ zip มี 30,000 ภาพ (รวม ~5 GB) ดาวน์โหลดจาก Hugging Face
- DFF ใช้โครงสร้างโฟลเดอร์แบบ IMDB-WIKI และภาพ fake ถูกสร้างจากภาพ/ชื่อของคนใน wiki →
  ถ้าชื่อไฟล์ตรงกันข้าม subset จะสุ่ม **ชุดชื่อไฟล์เดียวกัน** ทุก subset (paired design: คนเดียวกัน ต่างกันแค่ generator)
- extract เฉพาะไฟล์ที่สุ่มได้ ไม่แตก zip ทั้งหมด

In [ ]:
from huggingface_hub import hf_hub_download

IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

def get_zip(fname):
    p = ZIP_DIR / fname
    if p.exists():
        return p
    print("ดาวน์โหลด", fname)
    return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=fname, local_dir=str(ZIP_DIR)))

def zip_key(name):
    parts = name.split("/")
    rel = "/".join(parts[1:]) if len(parts) > 1 else name   # ตัดโฟลเดอร์บนสุด (wiki/, text2img/, ...)
    return os.path.splitext(rel)[0]

manifest_path = OUT_DIR / "sample_manifest.csv"
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    print("ใช้ manifest เดิม:", len(manifest), "ภาพ")
else:
    zip_index = {}
    for sub, (fname, _, _) in SUBSETS.items():
        zp = get_zip(fname)
        with zipfile.ZipFile(zp) as z:
            names = [n for n in z.namelist() if n.lower().endswith(IMG_EXTS) and "__MACOSX" not in n
                     and not Path(n).name.startswith(".")]
        zip_index[sub] = {zip_key(n): n for n in names}
        print(f"{sub:<11} {len(names):>6} ภาพ  ตัวอย่าง: {names[0] if names else '-'}")

    rng = random.Random(SEED)
    common = set.intersection(*[set(v) for v in zip_index.values()])
    n_target = N_PER_CLASS or min(len(v) for v in zip_index.values())
    PAIRED = len(common) >= n_target
    print(f"ชื่อไฟล์ที่ตรงกันทุก subset: {len(common)} → paired design = {PAIRED}")
    rows = []
    if PAIRED:
        keys = rng.sample(sorted(common), n_target)
        picks = {sub: keys for sub in SUBSETS}
    else:
        picks = {sub: rng.sample(sorted(idx), min(n_target, len(idx))) for sub, idx in zip_index.items()}
    for sub, (fname, label, pretty) in SUBSETS.items():
        out_dir = RAW_DIR / sub
        with zipfile.ZipFile(ZIP_DIR / fname) as z:
            for k in tqdm(picks[sub], desc=f"extract {sub}"):
                member = zip_index[sub][k]
                dst = out_dir / member
                if not dst.exists():
                    z.extract(member, out_dir)
                rows.append({"subset": sub, "label": label, "generator": pretty, "key": k,
                             "raw_path": str(dst), "paired": PAIRED})
        if DELETE_ZIPS_AFTER and ON_KAGGLE:
            (ZIP_DIR / fname).unlink(missing_ok=True)
    manifest = pd.DataFrame(rows)
    manifest.to_csv(manifest_path, index=False)
print(manifest.groupby("subset").size())

### 6.1 Crop หน้า (เก็บไฟล์ PNG แบบ lossless เพื่อไม่ให้เกิด JPEG artifact เพิ่ม)

In [ ]:
crop_meta_path = OUT_DIR / "crop_meta.csv"
if crop_meta_path.exists() and len(pd.read_csv(crop_meta_path)) == len(manifest):
    crop_meta = pd.read_csv(crop_meta_path)
else:
    recs = []
    t0 = time.time()
    for i, r in tqdm(manifest.iterrows(), total=len(manifest), desc="crop faces"):
        dst = CROP_DIR / r["subset"] / (r["key"].replace("/", "__") + ".png")
        rec = {"crop_path": str(dst)}
        if dst.exists():
            rec.update({"status": "ok"})
        else:
            rgb, info = crop_face(r["raw_path"])
            rec.update(info)
            if rgb is not None:
                dst.parent.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(dst), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
        recs.append(rec)
    crop_meta = pd.concat([manifest.reset_index(drop=True), pd.DataFrame(recs)], axis=1)
    crop_meta.to_csv(crop_meta_path, index=False)
    print(f"crop เสร็จใน {(time.time()-t0)/60:.1f} นาที")

agg = {"n": ("ok", "size"), "face_found": ("ok", "mean")}
if "is_gray" in crop_meta:
    crop_meta["is_gray"] = crop_meta["is_gray"].astype(float)
    agg["gray_frac"] = ("is_gray", "mean")      # สัดส่วนหน้าที่เป็นภาพขาวดำ (บอก domain shift)
det_summary = crop_meta.assign(ok=crop_meta["status"].eq("ok")).groupby("subset").agg(**agg)
print(det_summary.round(3))
det_summary.to_csv(OUT_DIR / "face_detection_summary.csv")

# ตัวอย่างหน้าที่ crop ได้ (แถวละ subset)
ok = crop_meta[crop_meta["status"] == "ok"]
# ใช้ key เดียวกันทุกแถว (ถ้าเป็น paired design จะเห็นคนเดียวกันจากแต่ละ generator)
key_sets = [set(ok[ok["subset"] == s]["key"]) for s in SUBSETS]
show_keys = sorted(set.intersection(*key_sets))[:6]
fig, axes = plt.subplots(len(SUBSETS), 6, figsize=(12, 2.2 * len(SUBSETS)))
for r_i, sub in enumerate(SUBSETS):
    sel = ok[ok["subset"] == sub]
    keys = show_keys if len(show_keys) == 6 else sel["key"].head(6).tolist()
    for c_i in range(6):
        ax = axes[r_i, c_i]
        ax.axis("off")
        if c_i < len(keys):
            ax.imshow(Image.open(sel[sel["key"] == keys[c_i]]["crop_path"].iloc[0]))
        if c_i == 0:
            ax.set_title(SUBSETS[sub][2], fontsize=9, loc="left")
plt.tight_layout()
plt.savefig(OUT_DIR / "crop_examples.png", dpi=120)
plt.show()

## 7. Inference

In [ ]:
class CropDataset(Dataset):
    def __init__(self, paths):
        self.paths = list(paths)
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        rgb = np.asarray(Image.open(self.paths[i]).convert("RGB"))
        if rgb.shape[:2] != (IMG_SIZE, IMG_SIZE):
            rgb = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        return to_tensor(rgb)

scores_path = OUT_DIR / "scores.csv"
eval_df = crop_meta[crop_meta["status"] == "ok"].reset_index(drop=True)
loader = DataLoader(CropDataset(eval_df["crop_path"]), batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
t0 = time.time()
outs = []
for x in tqdm(loader, desc="inference"):
    outs.append(predict_batch(x))
INFER_SEC = time.time() - t0
pred = {k: np.concatenate([o[k] for o in outs]) for k in outs[0]}
scores = pd.concat([eval_df, pd.DataFrame(pred)], axis=1)
scores.to_csv(scores_path, index=False)
print(f"inference {len(scores)} ภาพ ใน {INFER_SEC:.1f} s ({len(scores)/INFER_SEC:.0f} ภาพ/วินาที)")

## 8. ผลแยกตาม generator
แต่ละแถว = ภาพจริง (wiki) ทั้งหมด เทียบกับภาพปลอมของ generator นั้น  ·  "All fakes" = real vs fake ทุก generator รวมกัน (real:fake = 1:3)
95% CI ของ AUC ใช้ stratified bootstrap

In [ ]:
def bootstrap_auc_ci(y, s, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    y, s = np.asarray(y), np.asarray(s)
    i0, i1 = np.where(y == 0)[0], np.where(y == 1)[0]
    aucs = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(i0, len(i0)), rng.choice(i1, len(i1))])
        aucs.append(roc_auc_score(y[idx], s[idx]))
    return np.percentile(aucs, [2.5, 97.5])

FAKE_SUBS = [s for s, v in SUBSETS.items() if v[1] == 1]
REAL_SUBS = [s for s, v in SUBSETS.items() if v[1] == 0]
real = scores[scores["subset"].isin(REAL_SUBS)]
heads = ["main"] + (["fused", "branch2"] if DUAL else [])

def per_generator_table(head):
    rows = []
    groups = [(SUBSETS[g][2], scores[scores["subset"] == g]) for g in FAKE_SUBS]
    groups.append(("All fakes", scores[scores["subset"].isin(FAKE_SUBS)]))
    for name, fake in groups:
        d = pd.concat([real, fake])
        m = basic_metrics(d["label"], d[f"p_{head}"])
        lo, hi = bootstrap_auc_ci(d["label"].values, d[f"p_{head}"].values)
        rows.append({"head": head, "generator": name, **m, "AUC_CI_low": lo, "AUC_CI_high": hi})
    return pd.DataFrame(rows)

results = pd.concat([per_generator_table(h) for h in heads], ignore_index=True)
results.to_csv(OUT_DIR / "metrics_by_generator.csv", index=False)

show = results.copy()
for c in ["AUC", "AP", "EER", "ACC@0.5", "BalACC@0.5", "RealAcc(TNR)", "FakeAcc(TPR)", "AUC_CI_low", "AUC_CI_high"]:
    show[c] = (100 * show[c]).round(2)
show["AUC [95% CI]"] = show.apply(lambda r: f"{r['AUC']:.2f} [{r['AUC_CI_low']:.2f}, {r['AUC_CI_high']:.2f}]", axis=1)
print(show[["head", "generator", "n_real", "n_fake", "AUC [95% CI]", "AP", "EER", "ACC@0.5",
            "RealAcc(TNR)", "FakeAcc(TPR)"]].to_string(index=False))

### 8.1 ROC และการกระจายของ score (main branch)

In [ ]:
C_REAL, C_FAKE = "#2a78d6", "#eb6834"
C_GEN = {"text2img": "#2a78d6", "inpainting": "#eb6834", "insight": "#1baf7a"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({"axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

fig, ax = plt.subplots(figsize=(5.2, 5))
for g in FAKE_SUBS:
    d = pd.concat([real, scores[scores["subset"] == g]])
    fpr, tpr, _ = roc_curve(d["label"], d["p_main"])
    auc = roc_auc_score(d["label"], d["p_main"])
    ax.plot(fpr, tpr, lw=2, color=C_GEN.get(g), label=f"{SUBSETS[g][2]}  (AUC {100*auc:.1f})")
ax.plot([0, 1], [0, 1], ls="--", lw=1, color=INK2)
ax.set(xlabel="False positive rate", ylabel="True positive rate", xlim=(0, 1), ylim=(0, 1.01),
       title="ROC: CFB main branch, zero-shot on DFF")
ax.grid(color=GRID, lw=0.8)
ax.legend(loc="lower right", frameon=False)
plt.tight_layout()
plt.savefig(OUT_DIR / "roc_by_generator.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, len(FAKE_SUBS), figsize=(4.2 * len(FAKE_SUBS), 3.4), sharey=True)
bins = np.linspace(0, 1, 41)
for ax, g in zip(np.atleast_1d(axes), FAKE_SUBS):
    fake = scores[scores["subset"] == g]
    for vals, col, lab in [(real["p_main"], C_REAL, "Real (wiki)"), (fake["p_main"], C_FAKE, "Fake")]:
        ax.hist(vals, bins=bins, density=True, alpha=0.25, color=col)
        ax.hist(vals, bins=bins, density=True, histtype="step", lw=1.8, color=col, label=lab)
    ax.axvline(0.5, color=INK2, ls="--", lw=1)
    ax.set(title=SUBSETS[g][2], xlabel="P(fake)")
    ax.grid(axis="y", color=GRID, lw=0.8)
np.atleast_1d(axes)[0].set_ylabel("density")
np.atleast_1d(axes)[0].legend(frameon=False)
plt.tight_layout()
plt.savefig(OUT_DIR / "score_hist_by_generator.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 3.4))
data = [scores[scores["subset"] == s]["u_main"].values for s in SUBSETS]
ax.boxplot(data, showfliers=False, widths=0.5, medianprops={"color": C_FAKE, "lw": 2})
ax.set_xticks(range(1, len(SUBSETS) + 1), [SUBSETS[s][2] for s in SUBSETS], rotation=12)
ax.set(ylabel="uncertainty u = K/S", title="Main-branch uncertainty by subset")
ax.grid(axis="y", color=GRID, lw=0.8)
plt.tight_layout()
plt.savefig(OUT_DIR / "uncertainty_by_subset.png", dpi=150)
plt.show()

## 9. วิเคราะห์ threshold ตาม paper (Section III-D, Eq. 10–11, Table IX)
- **τ = 0.5** : threshold มาตรฐาน
- **τ_ot** : หา threshold ที่ให้ ACC สูงสุดบน *source* (ใช้ p ดิบ) แล้วนำไปใช้กับ *target*
- **τ_ot^UTOM** : เหมือน τ_ot แต่ปรับ p ก่อนด้วย `p' = tanh(λp / 2(u+ε))`, λ = 0.02, ε = 1e-7

Protocol: source = generator หนึ่งตัว, target = generator อื่น  ·  แบ่งภาพจริงเป็น 2 ครึ่ง (ครึ่ง A ใช้กับ source, ครึ่ง B ใช้กับ target)
เพื่อไม่ให้ภาพจริงชุดเดียวกันถูกใช้ทั้งหา threshold และวัดผล

In [ ]:
def utom_adjust(p, u, lam=UTOM_LAMBDA, eps=UTOM_EPS):
    return np.tanh(lam * p / (2 * (u + eps)))

def best_threshold(y, s):
    """Eq. 10–11: ลองทุกค่า score เป็น threshold แล้วเลือกตัวที่ ACC สูงสุด (ทำแบบ vectorized)"""
    y, s = np.asarray(y).astype(int), np.asarray(s, dtype=float)
    order = np.argsort(s)
    s_sorted, y_sorted = s[order], y[order]
    # ถ้า τ = s_sorted[i]: ทำนาย fake สำหรับ index ≥ i (กรณีค่าซ้ำ ใช้ index แรกของค่านั้น)
    real_below = np.concatenate([[0], np.cumsum(1 - y_sorted)])[:-1]
    fake_at_or_above = y_sorted.sum() - np.concatenate([[0], np.cumsum(y_sorted)])[:-1]
    acc = (real_below + fake_at_or_above) / len(y)
    first = np.r_[True, s_sorted[1:] != s_sorted[:-1]]
    i = np.argmax(np.where(first, acc, -1))
    return float(s_sorted[i]), float(acc[i])

def acc_at(y, s, t):
    return float(((np.asarray(s) >= t).astype(int) == np.asarray(y)).mean())

rng = np.random.default_rng(SEED)
real_idx = rng.permutation(real.index.values)
real_A, real_B = scores.loc[real_idx[: len(real_idx) // 2]], scores.loc[real_idx[len(real_idx) // 2:]]
scores["p_utom"] = utom_adjust(scores["p_main"].values, scores["u_main"].values)

rows = []
for src in FAKE_SUBS:
    S = pd.concat([real_A, scores[scores["subset"] == src]])
    S_putom = utom_adjust(S["p_main"].values, S["u_main"].values)
    t_ot, _ = best_threshold(S["label"], S["p_main"])
    t_utom, _ = best_threshold(S["label"], S_putom)
    for tgt in FAKE_SUBS:
        if tgt == src:
            continue
        T = pd.concat([real_B, scores[scores["subset"] == tgt]])
        T_putom = utom_adjust(T["p_main"].values, T["u_main"].values)
        rows.append({"source": SUBSETS[src][2], "target": SUBSETS[tgt][2],
                     "τ_ot": t_ot, "τ_ot^UTOM": t_utom,
                     "ACC@0.5": 100 * acc_at(T["label"], T["p_main"], 0.5),
                     "ACC@τ_ot": 100 * acc_at(T["label"], T["p_main"], t_ot),
                     "ACC@τ_ot^UTOM": 100 * acc_at(T["label"], T_putom, t_utom)})
thr_table = pd.DataFrame(rows)
thr_table.to_csv(OUT_DIR / "threshold_analysis.csv", index=False)
print(thr_table.round(4).to_string(index=False))
print("\nค่าเฉลี่ย ACC บน target:")
print(thr_table[["ACC@0.5", "ACC@τ_ot", "ACC@τ_ot^UTOM"]].mean().round(2).to_string())

## 10. บันทึกสรุป

In [ ]:
summary = {
    "config": {"N_PER_CLASS": N_PER_CLASS, "SEED": SEED, "IMG_SIZE": IMG_SIZE, "DET_THRES": DET_THRES,
               "USE_AMP": USE_AMP, "UTOM_LAMBDA": UTOM_LAMBDA, "N_BOOT": N_BOOT},
    "env": ENV, "repo_commit": REPO_COMMIT,
    "checkpoint": {"file": str(ckpt_file.name), "sha256": CKPT_SHA, "has_branch2": HAS_BRANCH2},
    "paired_design": bool(manifest["paired"].iloc[0]),
    "face_detection": det_summary.reset_index().to_dict(orient="records"),
    "metrics": results.to_dict(orient="records"),
    "threshold_analysis": thr_table.to_dict(orient="records"),
    "inference_seconds": INFER_SEC, "total_minutes": (time.time() - T_START) / 60,
}
with open(OUT_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=float)
print("ไฟล์ผลลัพธ์ใน", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print(" ", p.name)
print(f"เวลารวม {(time.time()-T_START)/60:.1f} นาที")

## หมายเหตุสำหรับเขียนรายงาน
- **Image-level vs video-level:** ตัวเลขใน paper เป็น video-level AUC (เฉลี่ย 8 frame) ส่วน DFF เป็นภาพนิ่ง → เป็น image-level AUC เทียบตรง ๆ ไม่ได้ทั้งหมด
- **Domain shift ฝั่งภาพจริง:** ภาพจริงของ DFF มาจาก IMDB-WIKI (ภาพถ่ายนิ่ง หลายยุค มีภาพขาวดำ ความละเอียดต่างกัน) ไม่ใช่ frame วิดีโอ YouTube แบบ FF++
  ดังนั้นค่า AUC สะท้อนทั้งความสามารถตรวจ fake และความต่างของ domain ฝั่ง real — ดู `RealAcc(TNR)` และ `gray_frac` ประกอบ
- **Preprocessing เท่ากันทุก subset:** ทุกภาพผ่าน RetinaFace + alignment + crop เดียวกัน; ภาพที่หาหน้าไม่เจอถูกตัดออก (ดู `face_detection_summary.csv`)
- **Threshold:** score ของโมเดลที่ฝึกด้วย evidential loss มักกระจุกใกล้ 0/1 → ACC@0.5 อาจต่ำแม้ AUC สูง ซึ่งเป็นเหตุผลที่ paper เสนอ UTOM
- **ข้อสังเกตต่อโค้ดต้นฉบับ:** `inference.py` ส่งภาพ BGR และอาจโหลด weight ไม่เข้าโดยไม่แจ้ง (filter เฉพาะ key `module.` + `strict=False`)